# Stage 2 Notebook 70 - Exp2OOO High-res mask aux (144x256) + NB62 recipe

**Architectural change #1: double the segmentation supervision resolution.** Across NB39-69 the aux mask was 72x128. CLRKDNet uses 288x800. Doubling our aux_mask_size to 144x256 forces the model's lane mask decoder + the feature pathway feeding it to encode FINER spatial detail. This propagates into the per-prior ROI features, which is what the cls head reads from.

Hypothesis: finer mask supervision tightens the spatial gradient signal, which improves the per-prior feature discrimination that has been the cls bottleneck.

Single architectural diff vs NB62 (exp57):
- `dataset.aux_mask_size: [72, 128] -> [144, 256]`
- `model.lane_head.mask_size: [72, 128] -> [144, 256]` (matches)
- `loss.lane.w_mask: 1.0 -> 1.5` (leverage the higher-res signal)
- This is an ARCHITECTURAL change, not just a hyperparameter knob: the mask decoder upsamples differently, the dataset loader produces a 144x256 GT mask, and the BCE+Dice loss runs on 4x more pixels.

Reference: HRNet (Sun et al. CVPR 2019) -- maintaining high-resolution representations improves fine-grained tasks. CLRKDNet itself uses 288x800 mask supervision.

### Run mode
1. Smoke.
2. 12 epochs full 70K. ~3-3.5 hr.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp65_rmt_gca_anchor_cls_sep_vfl_hires_mask_full_data_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp65_rmt_gca_anchor_cls_sep_vfl_hires_mask_full_data_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp65_rmt_gca_anchor_cls_sep_vfl_hires_mask_full_data_joint_smoke.log
OK exp65_rmt_gca_anchor_cls_sep_vfl_hires_mask_full_data_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.3433 det_loss=3.7727 grad_cos=0.1203 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4988110065460205, 'gate/lane_mean': 0.5022624135017395, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp65_rmt_gca_anchor_cls_sep_vfl_hires_mask_full_data_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full12'
    EPOCHS = 12
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp65_rmt_gca_anchor_cls_sep_vfl_hires_mask_full_data_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp65_rmt_gca_anchor_cls_sep_vfl_hires_mask_full_data_joint_full12 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp65_rmt_gca_anchor_cls_sep_vfl_hires_mask_full_data_joint_full12.tar --epochs 12 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp65_rmt_gca_anchor_cls_sep_vfl_hires_mask_full_data_joint_full12.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp65_rmt_gca_anchor_cls_sep_vfl_hires_mask_full_data_joint_full12_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stag

## What to watch in Exp2OOO

Reference NB62 (72x128 mask aux, w_mask=1.0): matched_iou=0.550, decoded_f1=0.073, val_lane_f1=0.118, val_lane_best_f1=0.138, val_lane_mask=0.45.

Pass criteria at epoch 12:
- val/lane/mask (BCE+Dice loss on the high-res mask) should DECREASE from ~0.55 to ~0.40 -- evidence the model learned fine-grained spatial features.
- val/matched_line_iou >= 0.55 (preserve NB62 geometry).
- val/lane/decoded_f1 >= 0.08 (10% over NB62).
- val/lane_f1 >= 0.13, val/lane_best_f1 >= 0.15.
- pos-neg gap >= 0.05.

If decoded_f1 jumps >= 0.10: high-res mask aux is the lift; queue for combining with topk_fixed and longer training.